# Getting started
## Introduction

This page gives a short overview over the basic usage and functionality of [magmeas](https://github.com/MaMMoS-project/magmeas). For this, we are going to use VSM-data from a publically available [example dataset uploaded to Nomad](https://nomad-lab.eu/prod/v1/gui/user/uploads/upload/id/dyT-vLgKTkqK5JeSy_HWwA/files/IFW_VSM_RawData).


## General concepts

The classes defined in `magmeas` should be understood as `mammos_entity.EntityCollection` with extra functionality suited for handling magnetometry data. It is worth the time to familiarise yourself with the [EntityCollection](https://mammos-project.github.io/mammos/api/mammos_entity.html#mammos_entity.EntityCollection) first.

The main class of magmeas is `Sweep` and handles the initialisation, some general calculations, IO-functionality and some basic plotting. All other classes inherit this behaviour and just add some additional functionality for specific use-cases.


## Initialisation

`Sweep` objects can be initialised as empty containers. New entities can always be added later. Alternatively, it is possible to pass entities as keyword arguments during the initialisation. As always, `mammos_entity.Entity` objects should be used wherever possible to not just incorporate numerical, but also dimensional and ontological information.

In [2]:
import mammos_entity as me

import magmeas

data = magmeas.Sweep(
    H_ext=me.Entity(
        "ExternalMagneticField",
        [-1e6, -0.8e6, -0.6e6, -0.5e6, -0.2e6, 0, 0.5e6, 1e6],
        "A/m",
    )
)
data["M"] = me.Entity(
    "Magnetization", [-1e6, -1e6, 1e6, 1e6, 1e6, 1e6, 1e6, 1e6], "A/m"
)
data

Sweep(
    H_ext=Entity(ontology_label='ExternalMagneticField', value=array([-1000000. -800000. -600000. ... 0. 500000. 1000000.]) (shape=(8,)), unit='A / m'),
    M=Entity(ontology_label='Magnetization', value=array([-1000000. -1000000. 1000000. ... 1000000. 1000000. 1000000.]) (shape=(8,)), unit='A / m'),
)

## Importing measurement data

`magmeas` was written with the analysis of **experimental** magnetometry data in mind. In order to do this, we'll have to get the experimental data into the classes of `magmeas` somehow. As you have just seen, this can always be done manually. Do your own reading and processing of data first and then assign it to the `magmeas` object.

However, some import functionality for common data files one might get from magnetometry equipment is already included in `magmeas`. Here is an example for an $M(H)$ measurement acquired on a 14 T PPMS from Quantum Design. The import methods are written so that they assign all of the data to the object they were called from and also return this object so that it can be quickly defined in just one line.

In [5]:
mh = magmeas.Sweep().read_PPMS("IFW_VSM_RawData/VSM_MH.DAT")
mh

Sweep(
    H_ext=Entity(ontology_label='ExternalMagneticField', value=array([12684.05213339 19174.8681775 29386.20953754 ... 3119.91434943 676.09019825 8.35563451]) (shape=(4864,)), unit='A / m'),
    mu=Entity(ontology_label='MagneticMoment', value=array([-0.00030001 -0.00027524 -0.00025582 ... 0.00383554 0.00381112 0.00377886]) (shape=(4864,)), unit='m2 A'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([300.0058289 300.0062714 300.0067139 ... 299.9986572 299.999115 299.9994354]) (shape=(4864,)), unit='K'),
    t=Entity(ontology_label='Time', value=array([13.95 14.65 15.83499999 ... 8834.23 8835.895 8850.035]) (shape=(4864,)), unit='s'),
)

More advanced import functionality which can be used to bundle together several steps and execute them at once is presented in more detail in a separate tutorial.

## Calculate Magnetisation

Since many data files only contain the `MagneticMoment` $\mu$, it is a common task to calculate the `Magnetization`. This can easily be done with the `calc_M` method all `magmeas` classes contain. Since the `Magnetization` is the `MagneticMoment` per unit volume, some additional information about the sample is needed. This can be assigned to the object before the calculation, but also while calling the `calc_M` method. The `MagneticPolarisation` is always calculated together with the `Magnetization`.

In [6]:
mh.calc_M(V=me.Entity("Volume", 10.64, "mm3"))
mh

Sweep(
    H_ext=Entity(ontology_label='ExternalMagneticField', value=array([12684.05213339 19174.8681775 29386.20953754 ... 3119.91434943 676.09019825 8.35563451]) (shape=(4864,)), unit='A / m'),
    mu=Entity(ontology_label='MagneticMoment', value=array([-0.00030001 -0.00027524 -0.00025582 ... 0.00383554 0.00381112 0.00377886]) (shape=(4864,)), unit='m2 A'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([300.0058289 300.0062714 300.0067139 ... 299.9986572 299.999115 299.9994354]) (shape=(4864,)), unit='K'),
    t=Entity(ontology_label='Time', value=array([13.95 14.65 15.83499999 ... 8834.23 8835.895 8850.035]) (shape=(4864,)), unit='s'),
    V=Entity(ontology_label='Volume', value=np.float64(10.64), unit='mm3'),
    M=Entity(ontology_label='Magnetization', value=array([-28196.70275641 -25868.67402397 -24042.81682863 ... 360482.92895278 358187.52889001 355156.30382131]) (shape=(4864,)), unit='A / m'),
    J=Entity(ontology_label='MagneticPolarisation', value=array([-0.03543302 -0.03250753 -0.03021309 ... 0.45299621 0.45011172 0.44630257]) (shape=(4864,)), unit='T'),
)

## Correct for demagnetisation

Another common task in the analysis of magnetometric data is the correction of the `ExternalMagneticField` for demagnetisation, yielding the `InternalMagneticField`. This can also be done with `magmeas`, using the `corr_demag` method. As before, the needed additional information, i.e. the `DemagnetizingFactor` can be assigned to the object before or while calling this method. For convenience, the sample information needed for calculating the `DemagnetizingFactor` of a rectangular cuboid can be used instead. 

In [11]:
mh.corr_demag(
    dimensions=me.Entity(
        "Length",
        [2.70, 1.99, 1.98],
        "mm",
        description="Direction of external magnetic field along first dimension.",
    )
)
mh

Sweep(
    H_ext=Entity(ontology_label='ExternalMagneticField', value=array([12684.05213339 19174.8681775 29386.20953754 ... 3119.91434943 676.09019825 8.35563451]) (shape=(4864,)), unit='A / m'),
    mu=Entity(ontology_label='MagneticMoment', value=array([-0.00030001 -0.00027524 -0.00025582 ... 0.00383554 0.00381112 0.00377886]) (shape=(4864,)), unit='m2 A'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([300.0058289 300.0062714 300.0067139 ... 299.9986572 299.999115 299.9994354]) (shape=(4864,)), unit='K'),
    t=Entity(ontology_label='Time', value=array([13.95 14.65 15.83499999 ... 8834.23 8835.895 8850.035]) (shape=(4864,)), unit='s'),
    V=Entity(ontology_label='Volume', value=np.float64(10.64), unit='mm3'),
    M=Entity(ontology_label='Magnetization', value=array([-28196.70275641 -25868.67402397 -24042.81682863 ... 360482.92895278 358187.52889001 355156.30382131]) (shape=(4864,)), unit='A / m'),
    J=Entity(ontology_label='MagneticPolarisation', value=array([-0.03543302 -0.03250753 -0.03021309 ... 0.45299621 0.45011172 0.44630257]) (shape=(4864,)), unit='T'),
    dimensions=Entity(ontology_label='Length', value=array([2.7 , 1.99, 1.98]), unit='mm', description='Direction of external magnetic field along first dimension.'),
    H=Entity(ontology_label='InternalMagneticField', value=array([20248.03555982 26114.3399048 35835.88096146 ... -93582.41725534 -95410.48260018 -95265.06761309]) (shape=(4864,)), unit='A / m'),
    B=Entity(ontology_label='MagneticFluxDensity', value=array([-0.00998859 0.00030871 0.0148196 ... 0.33539707 0.33021538 0.32658896]) (shape=(4864,)), unit='T'),
)

## Calculate magnetic flux density

Since we now have all the information we need, let's also calculate the `MagneticFluxDensity`.

In [8]:
mh.calc_B()
mh

Sweep(
    H_ext=Entity(ontology_label='ExternalMagneticField', value=array([12684.05213339 19174.8681775 29386.20953754 ... 3119.91434943 676.09019825 8.35563451]) (shape=(4864,)), unit='A / m'),
    mu=Entity(ontology_label='MagneticMoment', value=array([-0.00030001 -0.00027524 -0.00025582 ... 0.00383554 0.00381112 0.00377886]) (shape=(4864,)), unit='m2 A'),
    T=Entity(ontology_label='ThermodynamicTemperature', value=array([300.0058289 300.0062714 300.0067139 ... 299.9986572 299.999115 299.9994354]) (shape=(4864,)), unit='K'),
    t=Entity(ontology_label='Time', value=array([13.95 14.65 15.83499999 ... 8834.23 8835.895 8850.035]) (shape=(4864,)), unit='s'),
    V=Entity(ontology_label='Volume', value=np.float64(10.64), unit='mm3'),
    M=Entity(ontology_label='Magnetization', value=array([-28196.70275641 -25868.67402397 -24042.81682863 ... 360482.92895278 358187.52889001 355156.30382131]) (shape=(4864,)), unit='A / m'),
    J=Entity(ontology_label='MagneticPolarisation', value=array([-0.03543302 -0.03250753 -0.03021309 ... 0.45299621 0.45011172 0.44630257]) (shape=(4864,)), unit='T'),
    dimensions=Entity(ontology_label='Length', value=array([2.7 , 1.99, 1.98]), unit='mm'),
    H=Entity(ontology_label='InternalMagneticField', value=array([20248.03555982 26114.3399048 35835.88096146 ... -93582.41725534 -95410.48260018 -95265.06761309]) (shape=(4864,)), unit='A / m'),
    B=Entity(ontology_label='MagneticFluxDensity', value=array([-0.00998859 0.00030871 0.0148196 ... 0.33539707 0.33021538 0.32658896]) (shape=(4864,)), unit='T'),
)

## Finding entities

Like for a regular dictionary, you will always need the key of an entity to access and do something with it. If you want to know the key(s) of entities with a specific ontology label, you can use the method `get_entity_key` instead of searching through everything manually. This will return `None` if the ontology label could not be found, the key as a string if it could be found once and a list of strings if several entities in the object carry the specified ontology label.

In [13]:
print(mh.get_entity_key("ExternalMagneticField"))
print(mh.get_entity_key("Mass"))

H_ext
None
